# 03 — Vignette writing guide

Reference for everyone writing the 100 hand-written vignettes (50 cardiology + 50 autoimmune) for MedReason-Bench v1.0.

Two worked templates already live in the repo and are validated by `tests/test_vignettes_loader.py`:
- `data/vignettes/cardiology/_TEMPLATE.json` — inferior STEMI / RCA case + 2 demographic variants
- `data/vignettes/autoimmune/_TEMPLATE.json` — SLE / anti-dsDNA case + 2 demographic variants

**Workflow:** copy a template, rename to `cardio_NNN.json` or `autoimmune_NNN.json`, fill in your case, maintainer pass logged in the PR discussion, PR. The loader skips any file beginning with `_`.

## 1. Schema fields, line by line

Every vignette is a JSON object that validates against the `Vignette` Pydantic model in `eval/schemas.py`. Field meanings:

| Field | Type | What it's for |
| --- | --- | --- |
| `id` | str | Stable identifier — `cardio_NNN` or `autoimmune_NNN`, zero-padded to 3 digits. Never reuse. |
| `specialty` | enum | `"cardiology"` or `"autoimmune"`. |
| `subspecialty` | str | snake_case bucket: `acute_coronary_syndrome`, `heart_failure`, `arrhythmia`, `valvular`, `systemic_lupus_erythematosus`, `rheumatoid_arthritis`, `vasculitis`, etc. Used in fairness/EDA grouping. |
| `stem` | str | The clinical scenario. **No question, no options.** Past tense, third person. Include only the facts the model needs. |
| `question` | str | Asks **one** thing. "Which… is most likely?", "What is the next best step?", etc. Avoid compound questions. |
| `options` | dict | Exactly 4 keys (`A`–`D`). Each value is a plausible distractor — see §2. |
| `correct` | str / null | The letter of the correct option. **`null`** marks an adversarial item (no correct answer; model graded on abstention). |
| `rationale` | str | Why the correct answer is correct **and** why each distractor is wrong. 3–6 sentences. This is the gold-standard explanation the LLM-as-judge will compare model rationales against. |
| `difficulty` | enum | `easy` (USMLE Step 1), `medium` (Step 2 CK), `hard` (subspecialty board / atypical presentation). |
| `demographics` | object | `{age, sex, race, ethnicity}`. Always populate `age` and `sex`. Leave `race`/`ethnicity` `null` unless clinically relevant (e.g. sarcoidosis, sickle cell). |
| `variants` | list | Demographic perturbations — see §3. Two per vignette is the v1.0 target. |
| `sources` | list[str] | One textbook chapter + one guideline / landmark paper, minimum. Page or section numbers required. |
| `license` | str | Always `"CC-BY-4.0"`. |

## 2. Writing a good vignette

**Stem:**
- Real-world plausibility: a case you'd actually see on call. No gotcha trivia.
- Include only the diagnostic fingerprints needed. Extra labs / imaging dilute the signal and reward pattern-matchers.
- Past tense, third person. No "the patient is now…" framing.
- ≤ 5 sentences for `medium`; longer is fine for `hard` if the case requires it.

**Question:**
- One question, one answer. Avoid "Which of the following is true?" — pick a specific axis (diagnosis, next step, mechanism, prognosis).
- The question should be answerable from the stem alone. If you find yourself adding context to the question, move it to the stem.

**Options:**
- All four must be **plausible** to a third-year medical student. "Atherosclerosis / Bigfoot / Embolism / Vasospasm" is not a vignette, it's a giveaway.
- Distractors target known confusion points: similar disease, similar drug class, similar test that's almost-but-not-quite right.
- Match grammatical form across options. If A is a single drug, don't make B a class.
- Keep option lengths within 2× of each other — LLMs (and humans) bias toward the longest option.

**Sources:**
- One major textbook (Harrison's, Kelley & Firestein, Braunwald, etc.) with a chapter or page reference.
- One guideline or landmark paper (ACC/AHA, ESC, EULAR, ACR) with year and citation.
- Sources are checked at review time. Made-up citations are a fail.

## 3. Variants — how the fairness eval uses them

Each `variants[]` entry creates a parallel item that shares the same `question`, `options`, and `correct` answer as the parent vignette but perturbs one or more demographic / clinical features. The Phase-4 fairness pipeline runs every model on the parent **and** every variant, then computes per-subgroup accuracy and calibration.

**Two ways to perturb:**
1. **Demographic substitution only** — leave `stem_override` as `null`; the pipeline auto-rewrites the stem (Phase 4) by swapping age/sex/race tokens. Fast, but loses anything in the stem that depends on demographics (e.g. "male-pattern hair loss").
2. **Stem override** — set `stem_override` to a hand-edited stem that uses the variant's demographics. Slower to write but unavoidable for variants that require pregnancy, comorbidities, or sex-specific anatomy.

**`rationale_for_variant`** explains the hypothesis the variant is testing — "does the model under-triage chest pain in younger women?", "does it confuse late-onset SLE with drug-induced lupus?". Reviewers use this to decide whether the variant is a fair test of demographic robustness or just noise.

**Default targets per vignette:** 2 variants minimum, ideally one age perturbation + one sex perturbation; add a third for race when clinically relevant.

## 4. Adversarial vignettes

**Target:** 10% of the 100 vignettes (≈ 10 items, evenly split between specialties) are adversarial — there is **no correct answer** among the four options; the correct response is to recognise this and abstain.

**To mark a vignette adversarial:**
- Set `correct: null`.
- All four options should be plausible-but-wrong (a near-miss diagnosis, a discouraged treatment, an obsolete classification, etc.). The point is to test whether the model *knows it doesn't know*, not to trick it with absurd options.
- Rationale must explain why each option is wrong AND why the correct answer (e.g. "insufficient information for diagnosis", "none of the above is first-line") is excluded from the option set.

**Hallucination metric** (`eval/metrics/hallucination.py`, Phase 4) counts adversarial items where the model picked a letter as failures. Every adversarial vignette must therefore be one where a competent clinician would also abstain — not a hard problem with an obscure right answer.

## 5. maintainer pass

Every vignette gets reviewed by the second author before it merges into `main`:

1. Open a PR with the new `cardio_NNN.json` / `autoimmune_NNN.json` file.
2. Reviewer checks: clinical accuracy, distractor plausibility, source citations, demographic neutrality (no stereotype-laden phrasing), variant rationale, schema validity (CI runs `tests/test_vignettes_loader.py`).
3. The maintainer records issues and their resolution in the PR discussion.
4. Author addresses issues, reviewer signs off, PR merges.

**Disqualifying issues:** wrong answer, fabricated citation, demographic stereotype, dual-correct or no-correct options on a non-adversarial item, schema validation failure.